# Step 08 — Introduction to CrewAI

🇬🇧 **English** (this notebook)

A structured look at CrewAI itself, the framework the rest of this course builds on. Steps 02–07 explored prompting techniques on plain LLM calls; this is where `Agent`/`Task`/`Crew` enter the picture for real. We start with a plain LLM call again, watch it forget the conversation after one turn — the same problem every prompting-only step lived with — then bring in CrewAI to solve it properly.

This notebook covers only what you'll actually use for the rest of this course. CrewAI itself is much bigger — Flows, hierarchical processes, the full memory system, ~90 built-in tools, and more. For any of that, the [CrewAI documentation](https://docs.crewai.com) is the reference; the most relevant pages are linked under "Resources for further reading" at the end.

## Learning objective

By the end of this notebook, you will:

- Understand what CrewAI is and when to reach for it over a plain LLM call
- Understand CrewAI's core abstractions — `Agent`, `Task`, `Crew`, `Process` — and how they fit together
- Be able to work with `role`, `goal`, and `backstory` as CrewAI's version of prompts and system messages
- Have run a first `Agent`/`Task`/`Crew`, with `verbose=True` to see its reasoning
- Know every parameter `Agent(...)`, `Task(...)`, and `Crew(...)` accept, not just `role`/`goal`/`backstory`/`llm`, `description`/`expected_output`/`agent`, and `agents`/`tasks`/`process` — most left at their defaults here, but visible as a reference for when you need them later
- Have used `Task(output_pydantic=...)` to get a real Python object back instead of a string to parse yourself
- Have used CrewAI's built-in memory (`memory=True`) to give an agent multi-turn recall

## Prerequisites

- [Steps 02–07 — Prompting Techniques](step_02_zero_shot_prompting.ipynb) completed — this is where the course picks up CrewAI's framework, right after finishing plain-LLM-call prompting
- A working `.env` with a chat model API key (`GEMINI_API_KEY` or `OPENAI_API_KEY`), same as the previous steps
- No specific topic needed for this notebook itself — it uses its own running examples, not your team's topic, since it's about the framework rather than your project

## Background

CrewAI is a framework for orchestrating role-playing, autonomous AI agents that collaborate on shared tasks. It has no dependency on LangChain — it talks to any LLM provider directly through `litellm` (the same mechanism behind this repo's `MODEL` env var).

CrewAI ships two building blocks: **Crews** (teams of agents, optimized for flexibility and delegation) and **Flows** (event-driven, step-by-step control, closer to what a framework like LangGraph's graphs give you). **This course scopes itself to Crews only** — see the [CrewAI Flows docs](https://docs.crewai.com/en/concepts/flows) if you want to go further on your own.

## How this works

This notebook is one continuous walkthrough, not a single exercise cell — each section below builds on the previous one. Run the cells in order.

### A plain LLM call has no memory

Same idea as Step 01's Part 3 and Step 02 — a plain `crewai.LLM` call, no agent involved yet. Each call is an independent request/response round trip:

In [1]:
import os

from dotenv import load_dotenv
from crewai import LLM

load_dotenv()

llm = LLM(model=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"))

print(llm.call(messages=[{"role": "user", "content": "Hi, I am Tim!"}]))
print(llm.call(messages=[{"role": "user", "content": "What was my name again?"}]))

Hi Tim! It’s great to meet you. How are you doing today? Is there anything I can help you with?


As an AI, I don’t have access to your personal information, identity, or previous conversations unless you have provided that name in this specific chat session. 

If you told me your name earlier in this conversation, I may have "forgotten" if the context was cleared or if the session restarted. 

**What is your name?** I’d be happy to remember it for the rest of our chat!


### CrewAI's core abstractions

CrewAI structures agentic workflows around four building blocks — the same four labeled consistently across every notebook in this course from Step 09 onward:

- **`Agent`** — a `role`, `goal`, `backstory`, and (optionally) `tools` — CrewAI's version of a system prompt/persona
- **`Task`** — a `description` of the work assigned to an agent, and an `expected_output` describing what a good result looks like
- **`Crew`** — the collection of agents + tasks + a `process` for running them
- **`Process`** — the orchestration strategy: `sequential` (a fixed pipeline, used throughout this course) or `hierarchical` (a manager agent delegates dynamically — see the [Process docs](https://docs.crewai.com/en/concepts/processes) if you need it)

```
┌─────────┐
│  Start  │
└────┬────┘
     │
     ▼
┌──────────────┐
│ Agent + Task │
└──────┬───────┘
       │
       ▼
┌─────────┐
│   End   │
└─────────┘
```

When a `Crew` has multiple `Task`s, `Task(context=[other_task])` forwards a prior task's output into the next one — [Step 14](step_14_multi_agent_seq.ipynb) uses exactly this for handing work between agents.

In [2]:
from crewai import Agent, Task, Crew, Process

# ── Agent ─────────────────────────────────────────────────────────────────────
agent = Agent(
    role="Assistant",
    goal="Have a natural, helpful conversation with the user",
    backstory="You are a friendly, helpful assistant.",
    llm=llm,
    verbose=True,
)

# ── Task ──────────────────────────────────────────────────────────────────────
task = Task(
    description="Hi, I am Mei",
    expected_output="A natural, conversational reply to the user's message.",
    agent=agent,
)

# ── Crew ──────────────────────────────────────────────────────────────────────
crew = Crew(agents=[agent], tasks=[task], process=Process.sequential)

# ── Process — kick off the crew ────────────────────────────────────────────────
result = crew.kickoff()
print(result.raw)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Assistant                                                                                               │
│                                                                                                                 │
│  Task: Hi, I am Mei                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Assistant                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hi Mei! It's lovely to meet you. How are you doing today? Is there anything I can help you with?               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Hi Mei! It's lovely to meet you. How are you doing today? Is there anything I can help you with?


### The full list of `Agent` parameters

The example above only touched four of them: `role`, `goal`, `backstory`, `llm`. Here's everything else `Agent(...)` accepts, each shown at its actual default value — a reference for later, not something to memorize now. Same agent identity, same question, nothing else different.

In [3]:
# ── Agent — every parameter Agent(...) accepts. Only role/goal/backstory/llm
# are actually customized; everything else is shown at its default value,
# purely as a reference for what's available. ─────────────────────────────────
full_agent = Agent(
    role="Assistant",
    goal="Have a natural, helpful conversation with the user",
    backstory="You are a friendly, helpful assistant.",
    llm=llm,
    function_calling_llm=None,  # Default: None — optionally a separate, cheaper/faster LLM just for tool-calling decisions
    verbose=True,
    allow_delegation=False,  # Default: False — True lets this agent hand work off to another agent (Step 15)
    max_iter=25,  # Default: 25 — max reasoning steps before the agent must give its best answer
    max_rpm=None,  # Default: None — optionally a requests-per-minute cap, useful on a rate-limited free-tier key
    max_execution_time=None,  # Default: None — optionally a hard wall-clock timeout in seconds
    max_retry_limit=2,  # Default: 2 — retries on a failed LLM call before giving up
    allow_code_execution=False,  # Default: False — True lets the agent run Python it writes itself
    code_execution_mode="safe",  # Default: "safe" (sandboxed via Docker); "unsafe" runs directly on your machine
    respect_context_window=True,  # Default: True — automatically trims history that would exceed the model's context window
    use_system_prompt=True,  # Default: True — False folds role/goal/backstory into the user message instead
    multimodal=False,  # Default: False — True lets the agent accept image inputs, not just text
    inject_date=False,  # Default: False — True adds today's date to the prompt automatically
    date_format="%Y-%m-%d",  # Default: ISO format, only used when inject_date=True
    reasoning=False,  # Default: False — True turns on CrewAI's own plan-and-refine step before acting
    max_reasoning_attempts=None,  # Default: None — caps how many times reasoning=True can retry its own plan
    tools=[],  # Default: [] — e.g. [SerperDevTool()] for live web search, see Step 11
    knowledge_sources=None,  # Default: None — e.g. a TextFileKnowledgeSource for RAG, see Step 13
    embedder=None,  # Default: None — custom embedder config, needed alongside memory/knowledge_sources, see Step 10/13
    system_template=None,  # Default: None — optionally override how role/goal/backstory get assembled into the system prompt
    prompt_template=None,  # Default: None — optionally override the per-task prompt template
    response_template=None,  # Default: None — optionally override how the final answer gets formatted
    step_callback=None,  # Default: None — optionally a function called after every agent step, useful for logging/monitoring
)

full_task = Task(
    description="Hi, I am Mei",
    expected_output="A natural, conversational reply to the user's message.",
    agent=full_agent,
)

full_crew = Crew(agents=[full_agent], tasks=[full_task], process=Process.sequential)

full_result = full_crew.kickoff()
print(full_result.raw)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Assistant                                                                                               │
│                                                                                                                 │
│  Task: Hi, I am Mei                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Assistant                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hi Mei! It's lovely to meet you. How are you doing today? Is there anything I can help you with?               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Hi Mei! It's lovely to meet you. How are you doing today? Is there anything I can help you with?


### The full list of `Task` parameters

Same idea as the `Agent` reference above, now for `Task`. Only `description`, `expected_output`, and `agent` are actually customized below; everything else is shown at its default value. Source: [CrewAI's Task concept docs](https://docs.crewai.com/en/concepts/tasks).

In [4]:
# ── Task — every parameter Task(...) accepts. Only description/expected_output/
# agent are actually customized; everything else is shown at its default value,
# purely as a reference for what's available. ─────────────────────────────────
full_task = Task(
    description="Hi, I am Mei",
    expected_output="A natural, conversational reply to the user's message.",
    agent=full_agent,
    name=None,  # Default: None — an optional identifier for the task, useful for logging/lookup
    tools=[],  # Default: [] — restricts the agent to only these tools for this task; [] means "use whatever the agent already has"
    context=None,  # Default: auto-inferred from prior tasks in a sequential Crew when omitted — this task has none to chain from
    async_execution=False,  # Default: False — True runs this task in the background, in parallel with others
    human_input=False,  # Default: False — True pauses for a human review before the task is considered done (see Step 14)
    markdown=False,  # Default: False — True instructs the agent to format its final answer as Markdown
    config=None,  # Default: None — optional free-form dict for task-specific configuration
    output_file=None,  # Default: None — optionally write the task's output straight to a file
    create_directory=True,  # Default: True — create output_file's parent directory if it doesn't exist yet
    output_json=None,  # Default: None — optionally a Pydantic model to force structured JSON output
    output_pydantic=None,  # Default: None — optionally a Pydantic model the raw output gets parsed into
    callback=None,  # Default: None — optionally a function called after this task completes
    guardrail=None,  # Default: None — optionally a function that validates the output before the crew moves on
    guardrails=None,  # Default: None — optionally a list of guardrail functions, for more than one check
    guardrail_max_retries=3,  # Default: 3 — how many times a failed guardrail gets retried before giving up
)

# ── Crew — same agent as above, now with the fully-documented Task ───────────
full_crew = Crew(agents=[full_agent], tasks=[full_task], process=Process.sequential)

full_result = full_crew.kickoff()
print(full_result.raw)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Assistant                                                                                               │
│                                                                                                                 │
│  Task: Hi, I am Mei                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Assistant                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hi Mei! It’s a pleasure to meet you. I hope you're having a wonderful day so far. Is there anything on your    │
│  mind or anything I can help you with today?                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Hi Mei! It’s a pleasure to meet you. I hope you're having a wonderful day so far. Is there anything on your mind or anything I can help you with today?


### The full list of `Crew` parameters

Same idea again, now for `Crew`. Only `agents`, `tasks`, and `process` are actually customized below — already set in every example above; everything else is shown at its default value. Two of them are worth flagging up front because later sections in this notebook use them directly, not just as reference: `memory` (see "Giving it memory" below) and `tracing` (see "Seeing what happened: tracing" below). Source: [CrewAI's Crew concept docs](https://docs.crewai.com/en/concepts/crews).

In [5]:
# ── Crew — every parameter Crew(...) accepts. Only agents/tasks/process are
# actually customized; everything else is shown at its default value, purely
# as a reference for what's available. ────────────────────────────────────────
full_crew = Crew(
    agents=[full_agent],
    tasks=[full_task],
    process=Process.sequential,
    verbose=False,  # Default: False — True prints the same reasoning panels seen above, at the crew level
    cache=True,  # Default: True — caches tool call results so an identical call isn't repeated
    name="crew",  # Default: "crew" — a label, useful once you're running more than one Crew and telling logs apart
    memory=False,  # Default: False — True adds recall across separate kickoff() calls, see "Giving it memory" below
    embedder=None,  # Default: None — embedder config, required alongside memory=True or knowledge_sources
    knowledge_sources=None,  # Default: None — crew-wide knowledge sources, shared by every agent, see Step 13
    planning=False,  # Default: False — True runs an upfront planning pass: drafts a step-by-step plan per task and injects it into that Task's description before execution
    planning_llm=None,  # Default: None — LLM for the planning pass above; falls back to the crew's own agents' LLM if unset
    manager_llm=None,  # Default: None — required when process=Process.hierarchical, see Step 15
    manager_agent=None,  # Default: None — optionally your own Agent as manager instead of manager_llm, see Step 15
    function_calling_llm=None,  # Default: None — optionally a separate, cheaper/faster LLM for every agent's tool-calling decisions
    max_rpm=None,  # Default: None — crew-wide requests-per-minute cap, applies across all agents
    chat_llm=None,  # Default: None — set this to enable crew.chat(), an interactive Q&A mode over the crew
    stream=False,  # Default: False — True streams task output as it's generated instead of waiting for completion
    share_crew=False,  # Default: False — True shares full execution data (inputs/outputs) with CrewAI, used to improve their models
    step_callback=None,  # Default: None — optionally a function called after every agent step, across all agents
    task_callback=None,  # Default: None — optionally a function called after every task completes
    before_kickoff_callbacks=[],  # Default: [] — functions run on the inputs before kickoff, e.g. to validate or transform them
    after_kickoff_callbacks=[],  # Default: [] — functions run on the CrewOutput after kickoff, e.g. to post-process it
    output_log_file=None,  # Default: None — optionally write the run's log to this file
    prompt_file=None,  # Default: None — optionally override CrewAI's internal prompt templates via a JSON file
    tracing=None,  # Default: None — True/False explicitly enables/disables the dashboard from "Seeing what happened" below; None checks your environment/account settings
)

full_result = full_crew.kickoff()
print(full_result.raw)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Assistant                                                                                               │
│                                                                                                                 │
│  Task: Hi, I am Mei                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Assistant                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hi Mei! It’s a pleasure to meet you. I hope you're having a wonderful day so far. Is there anything on your    │
│  mind or anything I can help you with today?                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Hi Mei! It’s a pleasure to meet you. I hope you're having a wonderful day so far. Is there anything on your mind or anything I can help you with today?


### Structured output with Pydantic

Every example so far returned a plain string (`result.raw`) — fine for a chat reply, but "Sarah Johnson, sarah.johnson@techcorp.com, 555-0123" as a wall of text isn't something you can save to a database or a CSV row without parsing it yourself. `Task(output_pydantic=...)` fixes that: give it a Pydantic model, and the agent's final answer gets parsed straight into a real Python object.

Note for anyone coming from LangChain: CrewAI's `LLM` class has no `.with_structured_output()` method — `output_pydantic` on `Task` (already in the full-parameter list above, at its default of `None`) is CrewAI's own mechanism for the same idea.

In [6]:
from typing import Optional, List
from pydantic import BaseModel, Field

# ── Define the exact structure we want back ───────────────────────────────────
class Contact(BaseModel):
    """A single contact extracted from text."""
    first_name: str = Field(description="Person's first name")
    last_name: str = Field(description="Person's last name")
    phone: str = Field(description="Phone number in any format")
    email: Optional[str] = Field(default=None, description="Email address if mentioned")
    company: Optional[str] = Field(default=None, description="Company name if mentioned")

class ContactList(BaseModel):
    """List of contacts extracted from a document."""
    contacts: List[Contact] = Field(description="All contacts found in the text")

# ── Sample text — could just as easily come from a PDF or an inbox ───────────
sample_text = """
From the business meeting notes:

Sarah Johnson from TechCorp reached out regarding the partnership.
Her contact details are sarah.johnson@techcorp.com and 555-0123.

Also met with Michael Chen, mobile: (555) 0124. He's with DataSystems Inc.

Follow up with Jennifer Lopez at 555.0125, jlopez@example.com
"""

# ── Agent — a separate identity from the Assistant above, doesn't touch it ───
extractor = Agent(
    role="Contact Information Extractor",
    goal="Extract structured contact information from unstructured text with perfect accuracy",
    backstory="You are a meticulous data-entry specialist who never misses a name, phone number, or email buried in meeting notes.",
    llm=llm,
)

# ── Task — output_pydantic is what makes this structured. Distinct variable
# names throughout (contact_task/contact_crew/contact_result), so this doesn't
# overwrite the agent/task the memory section below still needs. ─────────────
contact_task = Task(
    description=(
        "Extract all person contact information from this text. Include first "
        "name, last name, phone number, email (if present), and company (if "
        f"mentioned).\n\nText:\n{sample_text}"
    ),
    expected_output="A structured list of every contact mentioned in the text.",
    agent=extractor,
    output_pydantic=ContactList,
)

contact_crew = Crew(agents=[extractor], tasks=[contact_task], process=Process.sequential)
contact_result = contact_crew.kickoff()

# ── contact_result.pydantic is a real ContactList instance — no string parsing ─
print(f"Extracted {len(contact_result.pydantic.contacts)} contacts:\n")
for contact in contact_result.pydantic.contacts:
    print(f"{contact.first_name} {contact.last_name} — {contact.phone}")
    if contact.email:
        print(f"  email: {contact.email}")
    if contact.company:
        print(f"  company: {contact.company}")

Extracted 3 contacts:

Sarah Johnson — 555-0123
  email: sarah.johnson@techcorp.com
  company: TechCorp
Michael Chen — (555) 0124
  company: DataSystems Inc.
Jennifer Lopez — 555.0125
  email: jlopez@example.com


#### Good to know

- `output_pydantic` and `output_json` both take the same kind of Pydantic model, but populate different attributes on the result: `output_pydantic` gives you `result.pydantic` (a real instance, `result.json_dict` is `None`); `output_json` gives you `result.json_dict` (a plain dict, `result.pydantic` is `None`). Pick whichever attribute access style you want to work with.
- `result.raw` is still there either way — the same JSON text, just not parsed for you.
- The model still has to *infer* the structure from unstructured text — it can still get a phone number or company wrong. Structured output guarantees the *shape* of the answer, not that every field is correct; you'd still spot-check this against the source text for anything that matters.

### Seeing what happened: tracing

The `verbose=True` log above is live but disposable — it's gone once the cell finishes, nothing to revisit or share afterward. `Crew(tracing=True)` is the alternative: it uploads that same information (agent reasoning, task timing, tool calls) to a free, no-signup dashboard and prints a link to it once the run finishes.

This notebook doesn't demo it live — the plain `llm.call()` at the very top means a later `Crew(tracing=True)` in this same session won't actually print a link, an unrelated quirk of running a plain LLM call before any `Crew` exists. You'll see it working for real starting in [Step 13](step_13_rag.ipynb), where every notebook opens straight with a `Crew`.

### Giving it memory

Same `Task`/`Crew` as above, with one thing added: `memory=True`. Without it, every `crew.kickoff()` starts fresh — nothing carries over to the next call, same problem as the plain LLM call at the top. With memory on, CrewAI stores what happened and retrieves it by semantic similarity next time, so a second, independent call below can recall Tim's name without being told again.

Memory needs an embedder (turns text into vectors for that similarity search) — point it at Gemini the same way the rest of this repo does. One gotcha: the embedder's model setting shares a "model" alias with this repo's own `MODEL` env var, so pin it explicitly via `EMBEDDINGS_GOOGLE_GENERATIVE_AI_MODEL_NAME`, or it silently inherits the wrong value:

In [7]:
os.environ.setdefault("EMBEDDINGS_GOOGLE_GENERATIVE_AI_MODEL_NAME", "gemini-embedding-001")

embedder = {
    "provider": "google-generativeai",
    "config": {"api_key": os.getenv("GEMINI_API_KEY")},
}

# ── Same Agent and Task as above — just memory=True and an embedder added to Crew ─
crew = Crew(agents=[agent], tasks=[task], process=Process.sequential, memory=True, embedder=embedder)
crew.reset_memories("all")  # start clean, ignoring memory from any earlier run of this notebook
print(crew.kickoff().raw)

# ── A second, independent call — the actual test. Nothing about Tim is restated ─
followup = Task(description="What was my name again?", expected_output="A natural, conversational reply.", agent=agent)
crew2 = Crew(agents=[agent], tasks=[followup], process=Process.sequential, memory=True, embedder=embedder)
print(crew2.kickoff().raw)

/Users/jgehbauer/Coding/research_crew/.venv/lib/python3.12/site-packages/chromadb/utils/embedding_functions/google_embedding_function.py:145: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Assistant                                                                                               │
│                                                                                                                 │
│  Task: Hi, I am Mei                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Assistant                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  It's so nice to meet you, Mei! How has your day been going so far? Is there anything on your mind that I can   │
│  help you with today?                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

It's so nice to meet you, Mei! How has your day been going so far? Is there anything on your mind that I can help you with today?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Assistant                                                                                               │
│                                                                                                                 │
│  Task: What was my name again?                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Assistant                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Your name is Mei! It’s lovely to chat with you again.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Your name is Mei! It’s lovely to chat with you again.


#### Good to know

- Memory is scoped **per agent `role`**, not per Python object or "conversation" — two `Crew`s reusing the same `Agent` share memory; two `Agent`s with different `role`s never do
- `crew.reset_memories("all")` clears it
- Recall is semantic/approximate (via the embedder), not an exact replay of past messages — don't rely on it for anything that needs to be verbatim-correct
- This is CrewAI's *short-term* memory. There's also long-term (SQLite, persists across restarts), entity (facts about named things), and external (bring-your-own backend, e.g. Mem0) — see the [CrewAI Memory docs](https://docs.crewai.com/en/concepts/memory) for all of it, including production/storage considerations

### Profile, Memory, Planning, Action: mapping the four modules to CrewAI

A common way to describe an LLM agent's architecture — used in the survey cited in [Step 09](step_09_single_agent.ipynb)'s "Resources for further reading" — splits it into four modules: Profile, Memory, Planning, Action. Here's where each one actually lives in what you just built above:

- **Profile** — `Agent(role=..., goal=..., backstory=...)`. These three strings are CrewAI's entire profiling mechanism; they get assembled into the system message. There's no separate persona object — the identity lives entirely in these three fields, unless you override the default assembly with `system_template`/`prompt_template`/`response_template`.
- **Memory** — `Crew(memory=True)` plus an `embedder`, from "Giving it memory" above. Short-term recall is scoped per agent `role`, retrieved by semantic similarity, not an exact transcript. CrewAI also ships long-term (SQLite), entity, and external memory — see the Memory docs linked below; this notebook only demos short-term.
- **Planning** — two separate opt-in mechanisms, not a standing "planner" component: `Crew(planning=True)` drafts a step-by-step plan per task before any agent starts (see the full `Crew` parameters cell above), while `Agent(reasoning=True)` runs its own plan-and-refine pass local to that one agent, independently of the crew-level flag. Neither keeps a persistent plan object the agent consults mid-task — once execution starts, the ReAct loop takes over.
- **Action** — the ReAct loop itself: `Agent(tools=[...])`, up to `max_iter` reasoning steps, calling a tool, observing the result, deciding whether to continue (Step 11 onward). `allow_code_execution=True` is a special case of Action — the agent runs Python it wrote itself, not just predefined tools. `Task(human_input=True)` ([Step 14](step_14_multi_agent_seq.ipynb)) is a brake on Action, not a fifth module — it pauses execution for a human check before a task counts as done.

One gap worth noting: CrewAI doesn't treat these four as equally-weighted, always-on modules. Profile and Action are the core of every `Agent`/`Task`; Memory and Planning are both opt-in flags layered on top — a simpler, more pragmatic architecture than the four-module framework might suggest at first glance.

## Your task

There's no team-topic exercise here — this notebook is about the framework itself. Instead:

1. Run every cell in order, from top to bottom, and read each output before moving to the next cell.
2. With `verbose=True` on, find the "🤖 Agent Started" and "✅ Agent Final Answer" panels in the log — that's CrewAI showing you what used to be invisible in the plain LLM call.
3. In the full-parameter `Agent` cell, try lowering `max_iter` to something small (e.g. `2`) and rerun — does the agent get cut off before finishing? Put it back afterward.
4. In the full-parameter `Task` cell, set `markdown=True` and rerun — does `full_result.raw` actually change, or does the agent already write markdown-flavored text without being told?
5. In the full-parameter `Crew` cell, set `planning=True` and rerun — CrewAI logs the plan it drafts before the agent starts on the task. Does the agent's final answer noticeably follow that plan, or does planning mostly just add latency for a task this simple?
6. Add a `role` field (e.g. `Optional[str]`, like "sales lead" or "engineer") to `Contact` and reword the Task's `description` to ask for it. Does the agent fill it in for every contact, or leave it `None` where the text doesn't say?
7. Change the Agent's `role`/`goal`/`backstory` and the Task's `description` to a persona and message of your choosing, then rerun the first example. How does the reply change?
8. Rerun the memory example with a different name and fact (e.g. "I am Alex and I work in marketing" / "What's my name and what do I do?"). Does recall still work?

## Shortcomings

Everything above was assembled directly in Python, inline, in one notebook cell each time — fine for learning, but not how you'd want to maintain a real project; and CrewAI's memory recall is semantic/approximate, not an exact replay of past turns.

This repo also ships a fuller reference project at `src/research_crew/`, where `role`/`goal`/`backstory` and task definitions live in YAML config instead of Python — see the main [README](../../README.md#the-template-code) for how that's organized. [Step 09](step_09_single_agent.ipynb) picks the `Agent` thread back up on a real research topic, and [Step 14](step_14_multi_agent_seq.ipynb) is where a second agent and a real `Crew` enter the picture for good, once Steps 10–13 have added memory, tools, MCP, and RAG to the single-agent line first.

## Resources for further reading

- [CrewAI documentation](https://docs.crewai.com) — the full concept reference (agents, tasks, processes, tools, memory, knowledge, flows)
- [CrewAI Flows docs](https://docs.crewai.com/en/concepts/flows) — the other half of CrewAI this course doesn't cover
- [CrewAI Memory docs](https://docs.crewai.com/en/concepts/memory) — the short-term/long-term/entity/external memory system introduced above
- [CrewAI Process docs](https://docs.crewai.com/en/concepts/processes) — `sequential` vs `hierarchical`, and the manager-agent pattern
- [CrewAI Tracing docs](https://docs.crewai.com/en/observability/tracing) — the `tracing=True` dashboard introduced above, including the persistent-account mode
- [CrewAI GitHub repository](https://github.com/crewAIInc/crewAI)

That's it for Step 08 — CrewAI's `Agent`/`Task`/`Crew`/`Process` are the exact same four abstractions every remaining notebook in this course uses, just with the topic and roles changed. On to [Step 09 — Single Agent](step_09_single_agent.ipynb).